# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. The notebook demonstrates programmatic access to tabular data and associated metadata via the Croissant schema.

### Dataset Source
The FAIR^2 dataset is defined by a Croissant schema available at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

This resource contains: clinical, demographic, pathological and molecular data about cancer survivors with second primary colorectal cancer.


In [ ]:
# Ensure `mlcroissant` is installed. Uncomment if running in a new environment.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display basic metadata (accessing as attributes)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. We'll enumerate record sets in the dataset, and for each, list its fields and columns if present.

In [ ]:
# List all record sets and their details by @id, following Croissant schema organization
record_sets = list(dataset.metadata.record_set)
print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"Record set name: {getattr(rs, 'name', '')}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    # List fields for this record set
    if rs.field:
        print("  Fields:")
        for f in rs.field:
            print(f"    - {f.id} (name: {getattr(f, 'name', '')}, dtype: {getattr(f, 'data_type', '')})")
    # List columns, if the record set references a fileObject with columns
    if hasattr(rs, 'file_object') and rs.file_object:
        for fo in rs.file_object:
            if hasattr(fo,'column') and fo.column:
                print("  Columns:")
                for c in fo.column:
                    print(f"    - {c.id} (name: {getattr(c, 'name', '')}, dtype: {getattr(c, 'data_type', '')})")
    print()

## 3. Data Extraction
Let's load the tabular data from the main record set to a DataFrame for processing. You will want to use the `@id` for the main (tabular) record set. We'll print all columns available.

In [ ]:
# For this dataset, let's assume there is a main record set with tabular data.
# We'll extract all record sets for generality, but focus on the primary table.

# -- Use the @id(s) as acquired from the previous cell --
# (Replace these IDs with actual values depending on printed output)
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} rows from record set: {record_set_id}")
            print(f"Columns: {list(df.columns)}\n")
        else:
            print(f"Record set {record_set_id} has no records.")
    except Exception as e:
        print(f"Failed loading records for record set {record_set_id}: {e}")
# --- For further steps, pick a specific record set ID as the main data table. ---
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nSample from '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular record sets found.")

## 4. Exploratory Data Analysis (EDA)
We'll explore, filter, and transform data based on its fields. For demonstration, we'll pick a numeric column (e.g., age) by its `@id` and show steps such as filtering and normalization.

> **Note:** All field/column names used below refer to their full `@id` unless stated otherwise.

In [ ]:
# --- Substitute your actual numeric and group field @id from the overview above ---
df = dataframes[main_record_set_id]

# Try several candidate field/column names that might represent age or other numeric properties
candidate_numeric_ids = [
    'Age', # generic
    'http://mlcommons.org/croissant/field/age',
    'age',
    'Age (years)',
    None
]

numeric_field_id = None
for col in df.columns:
    if col.lower() in ['age', 'age (years)', 'years']:
        numeric_field_id = col
        break
if not numeric_field_id:
    # Fallback: pick first numeric column
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break
print(f"Using numeric field: {numeric_field_id}")

# Example: Filter where numeric_field_id > 50 (e.g., age > 50)
threshold = 50
if numeric_field_id:
    filtered_df = df[df[numeric_field_id].astype(float, errors='ignore') > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field for filtered records
    field_norm = numeric_field_id + '_normalized'
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, field_norm]].head())
    # Try a group field; look for known categorical columns (e.g. sex, anatomical_location, etc.)
    group_candidates = [c for c in df.columns if 'sex' in c.lower() or 'anatom' in c.lower()]
    group_field_id = group_candidates[0] if group_candidates else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
    else:
        print("No obvious group/categorical field found for grouping.")
else:
    print("No numeric field detected in dataset for EDA.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric field, and also by group if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].astype(float, errors='ignore').dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # By group (if appropriate)
    if group_field_id:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
We have demonstrated how to connect to a FAIR^2 dataset defined by a Croissant schema, enumerate its record sets and fields by their `@id`, load tabular data, and perform basic EDA using Python's scientific libraries. This workflow can be adapted to any Croissant-compatible biomedical data resource using only programmatic identifiers.

- The dataset provides detailed clinical variables for 77 cancer survivors with second primary colorectal cancer.
- The primary tabular data is accessible and structured for statistics and ML.
- Using `mlcroissant`, all data elements are referenced via their stable `@id` for reproducibility.

For further analysis, consult the Croissant schema for precise definitions of each `@id`, and use this template to extract and analyze any subset of interest.